In [ ]:
from torch.nn import *
import torch
import numpy as np


class BatchNorm(BatchNorm1d):
    def forward(s, x):
        return super().forward(x.view(-1, x.size(-1))).view(x.shape)


def los(x, y):
    return functional.cross_entropy(
        x.permute(0, x.ndim - 1, *range(1, x.ndim - 1)), y.broadcast_to(x.shape[:-1])
    )


def acc(x, y):
    return (x.argmax(-1) == y).half().mean((-2,-1))


torch.set_default_tensor_type(torch.cuda.FloatTensor)
M = GRU(8, 320)
m = Sequential(
    Linear(324, 640, 0),
    BatchNorm(640, track_running_stats=0),
    Hardswish(1),
    Linear(640, 16),
)
o = torch.optim.Adam((*M.parameters(), *m.parameters()))
# M, m, o = torch.load("nnTrainerEvaler-Copy2.model.pk")
# torch.save((M, m, o), "nnTrainerEvaler-Copy2.model.pk")

In [ ]:
from sklearn.datasets import make_classification
import matplotlib.pyplot as plt


def axy():
    x = 16  # np.random.randint(2, 9)
    y = 1  # np.random.randint(1, 8 // x + 1)
    Y = 4  # np.random.randint(int(np.ceil(np.log2(x * y))), 5)
    a = [
        make_classification(
            512,
            4,
            n_informative=Y,
            n_redundant=np.random.randint(0, 5 - Y),
            n_classes=x,
            n_clusters_per_class=y,
        )
        for a in range(1)
    ]
    x = torch.tensor(np.stack([a[0] for a in a], 1), dtype=torch.float)
    a = np.stack([a[1] for a in a], 1)
    y = torch.tensor(a, dtype=torch.long)
    Y = torch.tensor(
        np.unpackbits(a.astype("B")[..., None], 2)[..., 4:], dtype=torch.float
    )
    # Y = functional.one_hot(y, 16).float()
    return torch.cat((x, Y), 2), x, y


def train(M, a):
    return M(a)[0]


def fast_eval(m, d, x, y):
    i = torch.randint(len(d), d.shape[:1]) % torch.arange(1, len(d) + 1)
    return torch.cat((d, x[i]), 2), y[i]


def slow_eval(m, d, x, y):
    i, I = torch.tril_indices(*d.shape[:1] * 2)
    return m(torch.cat((d[i], x[I]), 2)), y[I]


def full_eval(m, d, x, y):
    return (
        m(
            torch.cat(
                (
                    d[:, None].broadcast_to(len(d), *d.shape),
                    x.broadcast_to(len(d), *x.shape),
                ),
                3,
            )
        ),
        y,
    )


while 1:
    for i in range(5):
        for i in range(500):
            a, x, y = axy()
            d = train(M, a)
            x, y = slow_eval(m, d, x, y)
            i = los(x, y)
            i.backward()
            o.step()
            o.zero_grad()
        print(i, acc(x, y))
#     plt.figure()
#     plt.plot(acc(x, y).to("cpu", torch.half))
#     plt.show()